In [1]:
import pandas as pd
import numpy as np
import os

# 1. Define Paths
raw_path = r"C:\Users\Lenovo\Desktop\Superstore Sales Analytics\Data\Raw\Sales Dataset\train.csv"
processed_path = r"C:\Users\Lenovo\Desktop\Superstore Sales Analytics\Data\Processed\Cleaned_Superstore_Sales.csv"

# 2. Load Data
print("Loading raw data...")
df = pd.read_csv(raw_path)

# 3. Exploratory Data Analysis (EDA) Audit
print("\n--- Initial Data Audit ---")
print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")
print(f"Duplicate Rows: {df.duplicated().sum()}")
print(f"Missing Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

# 4. Data Cleaning & Type Conversion
print("\n--- Cleaning Data ---")
# Drop the useless Row ID
df.drop(columns=['Row ID'], inplace=True)

# Convert Dates (Format in CSV is DD/MM/YYYY)
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y')

# Handle missing Postal Codes
df['Postal Code'] = df['Postal Code'].fillna('Unknown')
df['Postal Code'] = df['Postal Code'].astype(str).apply(lambda x: x.replace('.0', '') if '.0' in x else x)

# Rename Sales to Revenue for business alignment
df.rename(columns={'Sales': 'Revenue'}, inplace=True)

# 5. Feature Engineering (Adding Business Value)
# Calculate Processing Time (Days between Order and Shipping)
df['Processing_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Extract Year and Month for Time-Series Analysis
df['Order_Year'] = df['Order Date'].dt.year
df['Order_Month'] = df['Order Date'].dt.month

# 6. Export to Processed Folder
df.to_csv(processed_path, index=False)
print(f"\nSuccess! Cleaned and engineered dataset saved to:\n{processed_path}")

Loading raw data...

--- Initial Data Audit ---
Total Rows: 9800 | Total Columns: 18
Duplicate Rows: 0
Missing Values:
Postal Code    11
dtype: int64

--- Cleaning Data ---

Success! Cleaned and engineered dataset saved to:
C:\Users\Lenovo\Desktop\Superstore Sales Analytics\Data\Processed\Cleaned_Superstore_Sales.csv


In [2]:
import pandas as pd
import numpy as np
import os

# Assume 'df' is your cleaned dataframe from the previous step
print("--- Commencing Advanced Data Science Features ---")

# ---------------------------------------------------------
# 1. OUTLIER DETECTION (Flagging "Whale" Orders using IQR)
# ---------------------------------------------------------
# Calculate Q1 and Q3 for Revenue
Q1 = df['Revenue'].quantile(0.25)
Q3 = df['Revenue'].quantile(0.75)
IQR = Q3 - Q1

# Define upper bound for what constitutes an abnormally large order
upper_bound = Q3 + 1.5 * IQR

# Create a new column flagging these anomalies
df['Is_Outlier_Order'] = np.where(df['Revenue'] > upper_bound, 'Yes (Whale)', 'No')

print(f"Detected {len(df[df['Is_Outlier_Order'] == 'Yes (Whale)'])} 'Whale' outlier orders.")

# Re-save the main file with the new Outlier column
processed_path = r"C:\Users\Lenovo\Desktop\Superstore Sales Analytics\Data\Processed\Cleaned_Superstore_Sales.csv"
df.to_csv(processed_path, index=False)


# ---------------------------------------------------------
# 2. RFM CUSTOMER SEGMENTATION (Data Science Modeling)
# ---------------------------------------------------------
print("\n--- Generating RFM Customer Segments ---")

# Set a snapshot date (1 day after the last order in the dataset)
snapshot_date = df['Order Date'].max() + pd.Timedelta(days=1)

# Group by Customer ID to calculate Recency, Frequency, and Monetary value
rfm = df.groupby('Customer ID').agg({
    'Order Date': lambda x: (snapshot_date - x.max()).days, # Recency
    'Order ID': 'nunique',                                  # Frequency
    'Revenue': 'sum'                                        # Monetary
}).reset_index()

# Rename columns for clarity
rfm.columns = ['Customer ID', 'Recency_Days', 'Frequency_Orders', 'Monetary_Total']

# Create scoring bins (1 to 4, where 4 is best)
# For Recency: lower days is better, so we reverse the labels
rfm['R_Score'] = pd.qcut(rfm['Recency_Days'], q=4, labels=[4, 3, 2, 1])
# For Frequency and Monetary: higher is better
rfm['F_Score'] = pd.qcut(rfm['Frequency_Orders'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary_Total'], q=4, labels=[1, 2, 3, 4])

# Combine into an RFM String (e.g., "444" is the best customer)
rfm['RFM_Segment_Code'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Assign readable Business Labels based on scores
def assign_segment(row):
    if row['R_Score'] == 4 and row['F_Score'] == 4 and row['M_Score'] == 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal Customers'
    elif row['R_Score'] == 1 and row['F_Score'] == 4:
        return 'Cant Lose Them (Churn Risk)'
    elif row['R_Score'] == 1 and row['F_Score'] == 1:
        return 'Lost/Inactive'
    else:
        return 'Standard/Regular'

rfm['Business_Segment'] = rfm.apply(assign_segment, axis=1)

# Save this as a SEPARATE table to build a Relational Model in Power BI
rfm_path = r"C:\Users\Lenovo\Desktop\Superstore Sales Analytics\Data\Processed\Customer_RFM_Segments.csv"
rfm.to_csv(rfm_path, index=False)

print(f"Success! RFM Segmentation table saved to:\n{rfm_path}")
print(rfm['Business_Segment'].value_counts())

--- Commencing Advanced Data Science Features ---
Detected 1145 'Whale' outlier orders.

--- Generating RFM Customer Segments ---
Success! RFM Segmentation table saved to:
C:\Users\Lenovo\Desktop\Superstore Sales Analytics\Data\Processed\Customer_RFM_Segments.csv
Business_Segment
Standard/Regular               438
Loyal Customers                207
Lost/Inactive                   93
Champions                       35
Cant Lose Them (Churn Risk)     20
Name: count, dtype: int64


In [3]:
import os
import shutil

# 1. Define your target directory
target_dir = r"C:\Users\Lenovo\Desktop\Superstore Sales Analytics\Python Notebook"

# 2. Create the folder if it does not already exist
os.makedirs(target_dir, exist_ok=True)

# 3. Define the current name of your notebook and the new path
# (Update "Untitled.ipynb" if your file has a different name right now)
current_notebook_name = "Sales.ipynb" 
destination_path = os.path.join(target_dir, "Superstore_Data_Science_EDA.ipynb")

# 4. Copy the notebook to the new directory
try:
    shutil.copy(current_notebook_name, destination_path)
    print(f"Success! Notebook saved to:\n{destination_path}")
except FileNotFoundError:
    print(f"Error: Could not find '{current_notebook_name}'. Please update the variable to match your current notebook name.")

Success! Notebook saved to:
C:\Users\Lenovo\Desktop\Superstore Sales Analytics\Python Notebook\Superstore_Data_Science_EDA.ipynb
